In [1]:
import pandas as pd
import numpy as np
import seaborn as sb
from collections import Counter
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv("telcoChurn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.drop(columns="customerID",inplace=True)

In [4]:
df.TotalCharges=pd.to_numeric(df.TotalCharges,errors='coerce')

In [5]:
df = df[df['TotalCharges'] > 0]

In [6]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


In [7]:
services = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

df['engagement_score'] = df[services].apply(
    lambda x: (x == 'Yes').sum(), axis=1
)
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 6, 12, 24, 72],
    labels=['0-6','6-12','12-24','24+']
)
df['high_value'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)

In [8]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn', 'engagement_score',
       'tenure_group', 'high_value'],
      dtype='object')

In [9]:
df['MultipleLines'] = df['MultipleLines'].replace(
    {'No phone service': 'No'}
).map({'Yes': 1, 'No': 0})
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
yes_no_cols = [
    'PhoneService','OnlineSecurity','OnlineBackup',
    'DeviceProtection','TechSupport','StreamingTV',
    'StreamingMovies','PaperlessBilling','Partner','Dependents','Churn'
]

for col in yes_no_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})


In [10]:
from sklearn.model_selection import train_test_split

x = df[['gender','SeniorCitizen','Partner','Dependents','tenure','InternetService','engagement_score','Contract','PaperlessBilling','PaymentMethod','MonthlyCharges']]
y = df['Churn']




In [11]:
num_cols = ['tenure', 'MonthlyCharges', 'engagement_score']
cat_cols = ['gender','SeniorCitizen','Contract','PaymentMethod',
            'InternetService']

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
        
    ],remainder='passthrough'
)

In [13]:
x=preprocessor.fit_transform(x)
pd.DataFrame(x)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,-1.280248,-1.161694,-0.562044,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
1,0.064303,-0.260878,-0.020634,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,-1.239504,-0.363923,-0.020634,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,0.512486,-0.747850,0.520776,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-1.239504,0.196178,-1.103454,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,-0.343137,0.664868,1.603596,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0
7028,1.612573,1.276493,1.062186,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0
7029,-0.872808,-1.170004,-0.562044,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0
7030,-1.158016,0.319168,-1.103454,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0


In [14]:
from sklearn.model_selection import train_test_split



x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [15]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [16]:
models={
    "SVM Classification":SVC(),
    "Decision Tree":DecisionTreeClassifier(),
    "Random Forest":RandomForestClassifier(),
    "AdaBoostClassifier":AdaBoostClassifier(),
    "Gradient Boost":GradientBoostingClassifier(),
    "XGboost":XGBClassifier()
}

for name,model in models.items():
    model.fit(x_train,y_train)
    ypred=model.predict(x_test)
    print(name)
    print(classification_report(y_test,ypred))

SVM Classification
              precision    recall  f1-score   support

           0       0.82      0.92      0.87      1549
           1       0.66      0.46      0.54       561

    accuracy                           0.79      2110
   macro avg       0.74      0.69      0.70      2110
weighted avg       0.78      0.79      0.78      2110

Decision Tree
              precision    recall  f1-score   support

           0       0.82      0.82      0.82      1549
           1       0.50      0.49      0.50       561

    accuracy                           0.74      2110
   macro avg       0.66      0.66      0.66      2110
weighted avg       0.73      0.74      0.73      2110

Random Forest
              precision    recall  f1-score   support

           0       0.82      0.89      0.85      1549
           1       0.60      0.47      0.53       561

    accuracy                           0.78      2110
   macro avg       0.71      0.68      0.69      2110
weighted avg       0.76    

In [17]:
model=XGBClassifier()
model.fit(x_train,y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [18]:
y_prob = model.predict_proba(x_test)[:, 1]

from sklearn.metrics import recall_score

thresholds = np.arange(0.1, 0.6, 0.05)
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    print(f"Threshold {t:.2f} → Recall: {recall_score(y_test, y_pred_t):.3f}")


Threshold 0.10 → Recall: 0.870
Threshold 0.15 → Recall: 0.834
Threshold 0.20 → Recall: 0.795
Threshold 0.25 → Recall: 0.743
Threshold 0.30 → Recall: 0.701
Threshold 0.35 → Recall: 0.642
Threshold 0.40 → Recall: 0.613
Threshold 0.45 → Recall: 0.563
Threshold 0.50 → Recall: 0.512
Threshold 0.55 → Recall: 0.456


In [19]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, (y_prob >= 0.35).astype(int))

array([[1232,  317],
       [ 201,  360]])

In [20]:
y_prob = model.predict_proba(x_test)[:, 1]
results = pd.DataFrame(x_test.copy())
results['actual_churn'] = y_test.values
results['churn_probability'] = y_prob*100



In [21]:
results

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,actual_churn,churn_probability
0,-1.280248,-1.512381,-1.103454,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,4.125269
1,-1.076528,-0.636495,-1.103454,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0,27.608097
2,1.571829,1.492555,2.145006,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0,1.099370
3,1.286622,0.177896,0.520776,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0,4.653364
4,0.471742,-0.109634,0.520776,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0,1.354245
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2105,0.553230,1.465963,1.603596,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0,38.209534
2106,1.571829,-1.495760,-1.103454,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0,0.020731
2107,-0.017185,0.872621,-0.020634,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0,7.272435
2108,-0.343137,-1.499084,-1.103454,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0,0.303694


In [22]:
Counter(results.actual_churn)

Counter({0: 1549, 1: 561})

In [23]:
def churn_risk_bucket(p):
    if p >= 70:
        return 'High Risk'
    elif p >= 40:
        return 'Medium Risk'
    else:
        return 'Low Risk'

results['risk_segment'] = results['churn_probability'].apply(churn_risk_bucket)


In [24]:
results['risk_segment'].value_counts(normalize=True)*100


risk_segment
Low Risk       70.331754
Medium Risk    17.914692
High Risk      11.753555
Name: proportion, dtype: float64

High Risk (≥ 0.7)

Extended warranty

Priority tech support

Discount on next product

Personalized call / email

 Goal: Prevent churn


Medium Risk (0.4–0.7)

App onboarding

Push notifications

Product education

Limited-time bundles

Goal: Increase engagement


Low Risk (< 0.4)

Recommend new product categories

Loyalty rewards

Upgrade suggestions

🎯 Goal: Increase NRR (115%+)